# VisML Lab: Time Series, Hands-On
DS-GA 3001, Fall 2026. Work in pairs. For each step: **predict first, then run.**

The data is real: the number of New York City taxi **passengers** counted in every 30-minute interval from July 1, 2014 to January 31, 2015 (10,320 rows). It comes from the [Numenta Anomaly Benchmark](https://github.com/numenta/NAB), which built it from the NYC Taxi and Limousine Commission trip records. Note the unit: the benchmark describes the values as passenger counts, not trips, and one trip can carry several passengers.

Three parts, following tonight's lecture:
1. **See the series**: the same data as a line, as daily totals, and as a heat map
2. **Evaluate a forecaster**: random split vs. time-ordered split
3. **Find anomalies**: a band around what is expected, and a threshold to choose

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score

plt.rcParams.update({"figure.figsize": (13, 4.5), "font.size": 12,
                     "axes.spines.top": False, "axes.spines.right": False})

URL = "https://raw.githubusercontent.com/numenta/NAB/master/data/realKnownCause/nyc_taxi.csv"
df = pd.read_csv(URL, parse_dates=["timestamp"]).rename(columns={"value": "passengers"})
df["date"] = df.timestamp.dt.normalize()
df["hour"] = df.timestamp.dt.hour + df.timestamp.dt.minute / 60    # 0, 0.5, 1, ... 23.5
df["dow"] = df.timestamp.dt.day_of_week                            # 0 = Monday
df["t"] = np.arange(len(df)) / 48                                  # days since July 1, 2014
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
print(len(df), "rows from", df.timestamp.min(), "to", df.timestamp.max())
df.head()

## Part 1. See the series

### Experiment 1: one line, two resolutions
**Predict before you run:** the first chart draws all 10,320 half-hour values as one line. What will you be able to read from it? A trend? A weekly pattern? Unusual days?

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.timestamp, df.passengers, lw=0.5, color="#57068c")
ax.set_ylabel("passengers per 30 minutes"); ax.set_title("Every 30-minute value, July 2014 to January 2015");

Now aggregate to **daily totals**. Same data, 215 points.

**Predict:** which days will stand out? Write down two dates you expect to be unusual for taxis in New York between July and January.

In [ ]:
daily = df.groupby("date").passengers.sum().to_frame("passengers")
daily["dow"] = daily.index.day_of_week

fig, ax = plt.subplots()
ax.plot(daily.index, daily.passengers, marker="o", ms=3, lw=1, color="#57068c")
ax.set_ylabel("passengers per day"); ax.set_title("Daily totals");

**Try:** change `START` to look at different weeks at full resolution. What does a normal week look like? Which day has the latest peak?

In [ ]:
START = "2014-09-08"                      # Sept 8th is a Monday; try other dates
week = df[(df.timestamp >= START) & (df.timestamp < pd.Timestamp(START) + pd.Timedelta(days=7))]
fig, ax = plt.subplots()
ax.plot(week.timestamp, week.passengers, color="#57068c")
ax.set_ylabel("passengers per 30 minutes"); ax.set_title(f"One week starting {START}");

### Experiment 2: a heat map of the whole series
One **row per day**, one **column per half hour**, color for the number of passengers. Nothing is aggregated: all 10,320 values are on screen.

**Predict:** what will a weekend look like in this picture? What will a snowstorm look like?

In [ ]:
grid = df.pivot(index="date", columns="hour", values="passengers")

fig, ax = plt.subplots(figsize=(13, 10))
im = ax.imshow(grid.values, aspect="auto", cmap="viridis", interpolation="nearest",
               extent=[0, 24, len(grid), 0])
ticks = [i for i, d in enumerate(grid.index) if d.day == 1]
ax.set_yticks(ticks); ax.set_yticklabels([grid.index[i].strftime("%b %d, %Y") for i in ticks])
ax.set_xticks(range(0, 25, 3)); ax.set_xlabel("hour of day")
ax.set_title("Passengers per 30 minutes: one row per day")
fig.colorbar(im, ax=ax, label="passengers per 30 minutes");

**Questions**
* A bright band on the left edge (midnight to 4 am) comes back every seven rows. What causes it?
* Find at least three rows that break the pattern. Use the tick labels to estimate their dates, then check them against the daily chart above.
* There is one very bright dash near November 1. Keep it in mind for Experiment 6.

The next cell shows the **average week**: the cyclic view of the same data.

In [ ]:
profile = df.groupby(["dow", "hour"]).passengers.mean().unstack()
fig, ax = plt.subplots()
for d in range(7):
    ax.plot(profile.columns, profile.loc[d], label=DAYS[d], lw=2.5 if d >= 5 else 1.3,
            color=plt.cm.tab10(d))
ax.set_xticks(range(0, 25, 3)); ax.set_xlabel("hour of day"); ax.set_ylabel("mean passengers per 30 minutes")
ax.set_title("Average day, by weekday (Saturday and Sunday drawn thicker)"); ax.legend(ncol=7, frameon=False);

## Part 2. Evaluate a forecaster

We train a gradient-boosted tree model to predict passengers per half hour, using July to September, and we want to know how well it will do in **October**, which it has never seen.

We estimate its error two ways before looking at October:
* **random 5-fold** cross-validation (rows shuffled)
* **time-ordered 5-fold** (`TimeSeriesSplit`: always train on the past, test on what follows)

### Experiment 3
The model gets `hour` and `dow`. With `USE_TIME_INDEX = True` it also gets `t`, the number of days since July 1.

**Predict:** with the time index included, which estimate will be closest to the real October error? Will the random estimate be too high or too low?

In [ ]:
USE_TIME_INDEX = True                     # run once with True, once with False

features = ["hour", "dow"] + (["t"] if USE_TIME_INDEX else [])
train = df[df.timestamp < "2014-10-01"]
october = df[(df.timestamp >= "2014-10-01") & (df.timestamp < "2014-11-01")]
make = lambda: HistGradientBoostingRegressor(random_state=0)

def cv_mae(cv):
    return -cross_val_score(make(), train[features], train.passengers, cv=cv,
                            scoring="neg_mean_absolute_error").mean()

mae_random = cv_mae(KFold(5, shuffle=True, random_state=0))
mae_ordered = cv_mae(TimeSeriesSplit(5))
model = make().fit(train[features], train.passengers)
mae_october = np.abs(model.predict(october[features]) - october.passengers).mean()

fig, ax = plt.subplots(figsize=(8, 4.5))
vals = [mae_random, mae_ordered, mae_october]
bars = ax.bar(["random 5-fold", "time-ordered 5-fold", "real error in October"], vals,
              color=["#e08214", "#57068c", "#6d6d6d"])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.0f}", ha="center", va="bottom", fontsize=14)
ax.set_ylabel("mean absolute error (passengers per 30 min)")
ax.set_title(f"Features: {features}");

**Questions**
* Run the cell again with `USE_TIME_INDEX = False`. What happens to the gap between the random estimate and the real October error?
* Why does a time index help the model so much on a random split?
* In October the model with the time index is the better one: its error is about a third lower (837 against 1,239). Which of the two models does **time-ordered** cross-validation prefer? Is that the right call?

The time-ordered estimate respects time, and it is still far from the October error. The next cell shows where its number comes from: the error on each of the five folds, with how much history each fold trained on. Run it with `USE_TIME_INDEX = True`.


In [ ]:
rows = []
for k, (itr, ite) in enumerate(TimeSeriesSplit(5).split(train), start=1):
    m = make().fit(train[features].iloc[itr], train.passengers.iloc[itr])
    err = np.abs(m.predict(train[features].iloc[ite]) - train.passengers.iloc[ite]).mean()
    rows.append({"fold": k, "days of training": round(len(itr) / 48),
                 "tested on": f"{train.date.iloc[ite[0]]:%b %d} to {train.date.iloc[ite[-1]]:%b %d}",
                 "MAE": round(err)})
print(f"mean of the five folds: {np.mean([r['MAE'] for r in rows]):.0f}    real error in October: {mae_october:.0f}")
pd.DataFrame(rows).set_index("fold")


**Questions**
* Which fold is responsible for most of the time-ordered average? What was different about the days it was tested on?
* The model you would deploy trains on 92 days and predicts the next 31. How much history does the first fold train on, and how long is each test window?
* A time-ordered split never trains on the future, so it is a *plausible* rehearsal of deployment. Is that the same as an *accurate* estimate of the October error? What would you change to make the rehearsal look more like the real thing?

### Experiment 4: what did the model miss?
This model only knows the **hour of day**. Its residuals (actual minus predicted) on October are grouped two ways.

**Predict:** what will the residuals look like by weekday?

In [ ]:
weak = make().fit(train[["hour"]], train.passengers)
res = october.assign(residual=october.passengers - weak.predict(october[["hour"]]))

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5))
a.boxplot([res.residual[res.dow == d] for d in range(7)], showfliers=False)
a.set_xticks(range(1, 8)); a.set_xticklabels(DAYS)
a.axhline(0, color="black", lw=1); a.set_title("Residuals by weekday"); a.set_ylabel("actual - predicted")
by = res.groupby(["dow", "hour"]).residual.mean().unstack()
im = b.imshow(by.values, aspect="auto", cmap="RdBu_r", vmin=-8000, vmax=8000, extent=[0, 24, 7, 0])
b.set_yticks(np.arange(7) + 0.5); b.set_yticklabels(DAYS); b.set_xticks(range(0, 25, 3)); b.set_xlabel("hour of day")
b.set_title("Mean residual by weekday and hour"); fig.colorbar(im, ax=b, label="actual - predicted");

**Questions**
* Where is the model too low (red) and too high (blue)? What feature would fix it?
* Would a single number (the MAE) have told you this?

## Part 3. Find anomalies

A simple detector on the **daily totals**:
1. *Expected* passengers for a day = the median for that weekday, learned from July to October
2. *Residual* = actual minus expected
3. Flag a day when its residual is more than `K` robust standard deviations from zero

### Experiment 5
**Predict:** with `K = 3`, how many days between July and January will be flagged? Will your two dates from Experiment 1 be among them?

In [ ]:
K = 3.0                                   # try 2, 3, 4, 5

ref = daily[:"2014-10-31"]
daily["expected"] = daily.dow.map(ref.groupby("dow").passengers.median())
daily["residual"] = daily.passengers - daily.expected
r = daily.residual[:"2014-10-31"]
sigma = 1.4826 * np.median(np.abs(r - r.median()))      # robust standard deviation
daily["z"] = daily.residual / sigma
flagged = daily[daily.z.abs() > K]

fig, ax = plt.subplots()
ax.fill_between(daily.index, daily.expected - K * sigma, daily.expected + K * sigma,
                color="#8900e1", alpha=0.18, lw=0, label=f"expected ± {K:g}σ")
ax.plot(daily.index, daily.passengers, color="#6d6d6d", lw=1.2, label="passengers per day")
ax.scatter(flagged.index, flagged.passengers, color="#c62828", zorder=3, label=f"flagged: {len(flagged)} days")
ax.axvline(pd.Timestamp("2014-11-01"), color="black", ls="--", lw=1)
ax.text(pd.Timestamp("2014-10-30"), ax.get_ylim()[0], "baseline learned here ", ha="right", va="bottom")
ax.set_ylabel("passengers per day"); ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 0.08))
flagged.assign(weekday=[DAYS[d] for d in flagged.dow])[["weekday", "passengers", "expected", "z"]].round(1)

The benchmark names five events in this series. The next cell checks **one calendar date for each**. We chose these dates; the benchmark does not give single dates.

In [ ]:
KNOWN = {"2014-11-01": "NYC Marathon weekend (race on Sunday, Nov 2)",
         "2014-11-27": "Thanksgiving",
         "2014-12-25": "Christmas",
         "2015-01-01": "New Year's Day",
         "2015-01-27": "blizzard"}
check = daily.loc[list(KNOWN), ["passengers", "expected", "z"]].round(1)
check["event"] = list(KNOWN.values())
check["flagged"] = check.z.abs() > K
check

One of the five dates, January 1, is **not** flagged at `K = 3`. Did the detector miss the New Year event?

The benchmark labels each event as a **window** of several days, not as a date. The next cell asks a different question of the same flags: for each window, was any day flagged, and how long after the window opened?


In [ ]:
# Anomaly windows for this series from the benchmark's labels file (labels/combined_windows.json)
WINDOWS = {"NYC Marathon":   ("2014-10-30 15:30", "2014-11-03 22:30"),
           "Thanksgiving":   ("2014-11-25 12:00", "2014-11-29 19:00"),
           "Christmas":      ("2014-12-23 11:30", "2014-12-27 18:30"),
           "New Year's Day": ("2014-12-29 21:30", "2015-01-03 04:30"),
           "blizzard":       ("2015-01-24 20:30", "2015-01-29 03:30")}
in_a_window = pd.Series(False, index=flagged.index)
rows = []
for name, (start, end) in WINDOWS.items():
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    hit = (flagged.index + pd.Timedelta(days=1) > start) & (flagged.index <= end)   # the day overlaps the window
    in_a_window |= hit
    days = flagged.index[hit]
    rows.append({"event": name, "window": f"{start:%b %d} to {end:%b %d}", "days flagged": len(days),
                 "first flag": f"{days[0]:%b %d}" if len(days) else "none",
                 "days after window opens": (days[0] - start.normalize()).days if len(days) else None})
print("flagged days outside every window:", ", ".join(f"{d:%b %d}" for d in flagged.index[~in_a_window]) or "none")
pd.DataFrame(rows).set_index("event")


**Questions**
* Change `K` to 2, then 4, then 5. How many days are flagged each time? How many of those would you want to be woken up for?
* Some flagged days are *before* November, inside the data the baseline was learned from. Which ones, and what happened then?
* Counting flagged **days**, counting **events** detected, and measuring the **delay** before the first flag are three different scores. Which one would you report for this detector, and why?
* January 1 itself was not flagged although its window was. The next experiment looks at that day at full resolution.

### Experiment 6: the anomaly that the daily total hides
**Predict:** New Year's Day has a nearly normal daily total. What will it look like half hour by half hour?

Then set `DAY = "2014-11-02"`. The largest value in the whole series is on that night. What do you think happened at 2 am on November 2, 2014? That is a hypothesis: what evidence would confirm it, and can this file alone provide it?

In [ ]:
DAY = "2015-01-01"                        # then try 2014-11-02, 2015-01-27, 2014-07-04
base = df[df.timestamp < "2014-11-01"].groupby(["dow", "hour"]).passengers.median()
one = df[df.date == DAY]
expected = base.loc[one.dow.iloc[0]]

fig, ax = plt.subplots()
ax.plot(expected.index, expected.values, color="#6d6d6d", ls="--", label=f"a typical {DAYS[one.dow.iloc[0]]}")
ax.plot(one.hour, one.passengers, color="#c62828", lw=2.5, label=DAY)
ax.set_xticks(range(0, 25, 3)); ax.set_xlabel("hour of day"); ax.set_ylabel("passengers per 30 minutes")
ax.legend(frameon=False);

## Debrief
* Which view showed you the unusual days fastest: the line, the daily totals, or the heat map?
* The daily total for January 1 looked ordinary, although the day was not. What does that say about choosing the bin size for a monitoring chart?
* The random split reported an error a third lower than the real one, and the time-ordered split preferred the wrong model. Where in your own project could the same thing happen?
* If this were a live system, what would you put on the dashboard: the raw series, the band, the flagged list, or all three?

**Going further:** replace the weekday median with the model from Part 2 as the "expected" value, and run the detector on half-hour data. Does it find the marathon?